# Colab bootstrap
Checks that Colab can see the code, Drive and ngspice, and that Colab's ngspice agrees with the Mac's.

**Before running:** push your latest code to GitHub, and (if data changed) wait for Drive for desktop to finish uploading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = "/content/spice-verified-analog-finetuning"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Keerthanareddy17/spice-verified-analog-finetuning.git", REPO], check=True)
os.chdir(REPO)

In [ ]:
!apt-get -qq install -y ngspice > /dev/null
!ngspice --version | head -3
!pip -q install -e .

In [ ]:
import json
from analogft import paths
from analogft.circuit import nominal_design, simulate
print("IS_COLAB:", paths.IS_COLAB, "| STORAGE_ROOT:", paths.STORAGE_ROOT)

ref = json.load(open(paths.REPO_ROOT / "results" / "reference_nominal.json"))
r = simulate(nominal_design(), use_cache=False)
print("Mac ngspice:", ref["ngspice_version"], "| Colab ngspice:", r.ngspice_version)
for k, v in ref["metrics"].items():
    c = r.metrics[k]
    rel = abs(c - v) / max(abs(v), 1e-30)
    print(f"{k:<11} mac={v:<14.6g} colab={c:<14.6g} rel.diff={rel:.2e}", "OK" if rel < 1e-2 else "MISMATCH")
assert r.ok and all(abs(r.metrics[k]-v)/max(abs(v),1e-30) < 1e-2 for k, v in ref["metrics"].items()), "Colab ngspice disagrees with the Mac"
print("Colab ngspice agrees with the Mac (within 1%).")

In [ ]:
# Drive data check: raises if Drive for desktop has not finished uploading.
from analogft.drive_check import verify_manifest
for name in ("data", "runs", "results"):
    root = paths.STORAGE_ROOT / name
    if not (root / "manifest.json").exists():
        print(name, ": no manifest yet"); continue
    verify_manifest(root, raise_on_error=True)
    print(name, ": complete")
